# 📊 NY Lottery Data Exploration

This notebook explores the NY Lottery dataset, examining data quality, structure, and basic statistics.

## Objectives
- Understand the data structure
- Examine data quality and completeness
- Generate summary statistics for all games
- Visualize basic distributions

## ⚠️ Disclaimer
This analysis is for educational purposes only. Lottery games are random and independent events. No analysis can predict future outcomes.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from pathlib import Path

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

# Database paths
DB_PATH = Path('data/lottery_star.db')
DB_SIMPLE = Path('data/lottery.db')

print("Libraries loaded successfully!")

## 1. Database Connection & Schema Overview

In [ ]:
# Connect to the star schema database
conn = sqlite3.connect(DB_PATH)

# List all tables
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(tables_query, conn)
print("Tables in lottery_star.db:")
print(tables)

In [ ]:
# Examine table schemas
for table in tables['name']:
    print(f"\n{'='*60}")
    print(f"Table: {table}")
    print('='*60)
    schema_query = f"PRAGMA table_info({table});"
    schema = pd.read_sql_query(schema_query, conn)
    print(schema[['name', 'type', 'pk']].to_string(index=False))

## 2. Game Dimension Analysis

In [ ]:
# Get game information
games_df = pd.read_sql_query("SELECT * FROM dim_game", conn)
print("Games in database:")
print(games_df)

In [ ]:
# Draw counts per game
draw_counts = pd.read_sql_query('''
    SELECT 
        dg.game_name,
        dg.game_type,
        COUNT(DISTINCT fsd.draw_date) as total_draws,
        MIN(fsd.draw_date) as first_draw,
        MAX(fsd.draw_date) as last_draw
    FROM dim_game dg
    LEFT JOIN fact_set_draws fsd ON dg.game_id = fsd.game_id
    WHERE dg.game_type = 'set_draw'
    GROUP BY dg.game_id
    ORDER BY total_draws DESC
''', conn)
print("Set-Draw Games Summary:")
print(draw_counts)

In [ ]:
# Visualize draw counts
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(draw_counts['game_name'], draw_counts['total_draws'], color=sns.color_palette('viridis', len(draw_counts)))
ax.set_xlabel('Number of Draws')
ax.set_title('Total Draws by Game')
for bar, count in zip(bars, draw_counts['total_draws']):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2, f'{count:,}', va='center')
plt.tight_layout()
plt.show()

## 3. Date Dimension Analysis

In [ ]:
# Analyze date dimension
date_df = pd.read_sql_query("SELECT * FROM dim_date", conn)
date_df['date'] = pd.to_datetime(date_df['date'])
print(f"Date range: {date_df['date'].min()} to {date_df['date'].max()}")
print(f"Total unique dates: {len(date_df):,}")

In [ ]:
# Draws by weekday
weekday_counts = pd.read_sql_query('''
    SELECT dd.weekday, COUNT(*) as draw_count
    FROM fact_set_draws fsd
    JOIN dim_date dd ON fsd.draw_date = dd.date
    GROUP BY dd.weekday
''', conn)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_counts['weekday'] = pd.Categorical(weekday_counts['weekday'], categories=day_order, ordered=True)
weekday_counts = weekday_counts.sort_values('weekday')

plt.figure(figsize=(10, 5))
plt.bar(weekday_counts['weekday'], weekday_counts['draw_count'], color=sns.color_palette('coolwarm', 7))
plt.xlabel('Day of Week')
plt.ylabel('Number of Draws')
plt.title('Lottery Draws by Day of Week (All Games)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Number Frequency Overview

In [ ]:
# Get all numbers from fact_set_numbers
numbers_df = pd.read_sql_query('''
    SELECT dg.game_name, fsn.number, COUNT(*) as frequency
    FROM fact_set_numbers fsn
    JOIN dim_game dg ON fsn.game_id = dg.game_id
    GROUP BY dg.game_name, fsn.number
    ORDER BY dg.game_name, frequency DESC
''', conn)
print(f"Total number records: {len(numbers_df):,}")
print("\nSample data:")
print(numbers_df.head(20))

In [ ]:
# Frequency distribution for each game
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

games = numbers_df['game_name'].unique()
for i, game in enumerate(games[:6]):
    game_nums = numbers_df[numbers_df['game_name'] == game]
    axes[i].bar(game_nums['number'], game_nums['frequency'], alpha=0.7)
    axes[i].set_title(f'{game.replace("_", " ").title()}')
    axes[i].set_xlabel('Number')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Number Frequency Distribution by Game', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Daily Games Analysis

In [ ]:
# Analyze daily games (Numbers and Win4)
daily_df = pd.read_sql_query('''
    SELECT 
        dg.game_name,
        fdd.session_id,
        ds.session_name,
        COUNT(*) as total_draws
    FROM fact_daily_draws fdd
    JOIN dim_game dg ON fdd.game_id = dg.game_id
    JOIN dim_session ds ON fdd.session_id = ds.session_id
    GROUP BY dg.game_name, ds.session_name
''', conn)
print("Daily Games Summary:")
print(daily_df)

In [ ]:
# Digit frequency analysis for Numbers game
numbers_picks = pd.read_sql_query('''
    SELECT fdd.pick
    FROM fact_daily_draws fdd
    JOIN dim_game dg ON fdd.game_id = dg.game_id
    WHERE dg.game_name = 'numbers'
''', conn)

# Analyze digit frequency by position
digit_freq = {1: {}, 2: {}, 3: {}}
for pick in numbers_picks['pick']:
    if len(pick) == 3:
        for pos, digit in enumerate(pick, 1):
            digit_freq[pos][digit] = digit_freq[pos].get(digit, 0) + 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for pos in range(1, 4):
    digits = sorted(digit_freq[pos].keys())
    counts = [digit_freq[pos][d] for d in digits]
    axes[pos-1].bar(digits, counts)
    axes[pos-1].set_title(f'Position {pos}')
    axes[pos-1].set_xlabel('Digit')
    axes[pos-1].set_ylabel('Frequency')

plt.suptitle('Digit Frequency by Position (Numbers Game)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Data Quality Summary

In [ ]:
# Check for missing values and data quality
print("Data Quality Report")
print("=" * 60)

# Check fact_set_draws
fsd_nulls = pd.read_sql_query('''
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN bonus IS NULL OR bonus = '' THEN 1 ELSE 0 END) as null_bonus
    FROM fact_set_draws
''', conn)
print(f"\nfact_set_draws:")
print(f"  Total rows: {fsd_nulls['total_rows'].iloc[0]:,}")
print(f"  Null/empty bonus: {fsd_nulls['null_bonus'].iloc[0]:,}")

# Check fact_set_numbers
fsn_stats = pd.read_sql_query('''
    SELECT 
        COUNT(*) as total,
        MIN(number) as min_num,
        MAX(number) as max_num,
        AVG(number) as avg_num
    FROM fact_set_numbers
''', conn)
print(f"\nfact_set_numbers:")
print(f"  Total records: {fsn_stats['total'].iloc[0]:,}")
print(f"  Number range: {fsn_stats['min_num'].iloc[0]} - {fsn_stats['max_num'].iloc[0]}")
print(f"  Average number: {fsn_stats['avg_num'].iloc[0]:.2f}")

conn.close()
print("\n✅ Data quality check complete!")

## Summary

This notebook explored the NY Lottery database structure and generated basic statistics:

1. **Database Schema**: The star schema contains dimension tables (game, date, session) and fact tables (draws, numbers)
2. **Game Coverage**: Multiple games including Mega Millions, Powerball, Take 5, Cash4Life, NY Lotto, Numbers, and Win4
3. **Data Volume**: Thousands of historical draws dating back several years
4. **Data Quality**: The data appears clean with minimal missing values

**Next Steps:**
- Proceed to `02_frequency_analysis.ipynb` for detailed frequency analysis
- Explore `03_simulations.ipynb` for Monte Carlo simulations